<!-- TARGET: public course-materials repo, sessions/m04/demo.ipynb (full version). Copy to work/m04/ before editing. -->
# Module 4 · Live demo

**Python for Data Science · ESADE MIBA · Wed 4 Nov 2026**

The marketplace operations team asks: *how fast do we deliver, and how often are we late?*
Before we can answer, we need Python's building blocks, then pandas, then a clean look at the
orders table.

## 1. Python through data

Six orders from a tiny shop, the same ones as in the warm-up. A **list of dictionaries**: each
dictionary is one row, each key is a column.

In [1]:
orders = [
    {"order_id": "o1", "customer_state": "SP", "price": 129.90, "freight": 15.10, "purchased": 2, "delivered": 9, "estimated": 12},
    {"order_id": "o2", "customer_state": "RJ", "price": 49.90, "freight": 12.00, "purchased": 3, "delivered": 20, "estimated": 15},
    {"order_id": "o3", "customer_state": "SP", "price": 89.00, "freight": 8.50, "purchased": 5, "delivered": 10, "estimated": 14},
    {"order_id": "o4", "customer_state": "MG", "price": 199.00, "freight": 22.30, "purchased": 6, "delivered": None, "estimated": 18},
    {"order_id": "o5", "customer_state": "SP", "price": 35.50, "freight": 7.90, "purchased": 8, "delivered": 25, "estimated": 20},
    {"order_id": "o6", "customer_state": "RJ", "price": 74.90, "freight": 11.20, "purchased": 9, "delivered": 16, "estimated": 19},
]
len(orders)

6

### Lists: order, position, length

In [2]:
states = ["SP", "RJ", "SP", "MG", "SP", "RJ"]
print(states[0], states[-1], states[1:3], len(states))

SP RJ ['RJ', 'SP'] 6


### Dictionaries: look up by name, not by position

In [3]:
first_order = orders[0]
print(first_order["price"])
first_order["review_score"] = 5
first_order.keys()

129.9


dict_keys(['order_id', 'customer_state', 'price', 'freight', 'purchased', 'delivered', 'estimated', 'review_score'])

### Loops: the accumulator pattern

*Your turn (1 min):* before running, say out loud what `total_revenue` will be after the first
pass of the loop.

In [4]:
total_revenue = 0
for order in orders:
    total_revenue = total_revenue + order["price"]
total_revenue

578.2

### `if` inside a loop, and the `None` trap

In [5]:
late_ids = []
for order in orders:
    if order["delivered"] is not None and order["delivered"] > order["estimated"]:
        late_ids.append(order["order_id"])
late_ids

['o2', 'o5']

*Your turn (2 min):* what happens if we drop `order["delivered"] is not None and`? Predict, then I run it.

### Functions: name a rule once, reuse it everywhere

In [6]:
def is_late(order):
    return order["delivered"] is not None and order["delivered"] > order["estimated"]


def delivery_days(order):
    if order["delivered"] is None:
        return None
    return order["delivered"] - order["purchased"]


[delivery_days(order) for order in orders]

[7, 17, 5, None, 17, 7]

### A group-by, by hand

In [7]:
revenue_by_state = {}
for order in orders:
    state = order["customer_state"]
    if state not in revenue_by_state:
        revenue_by_state[state] = 0
    revenue_by_state[state] = revenue_by_state[state] + order["price"]
revenue_by_state

{'SP': 254.4, 'RJ': 124.80000000000001, 'MG': 199.0}

### The same table in pandas

A **DataFrame** is a table object. It has **attributes** (data you read: `df.shape`, no
brackets) and **methods** (actions you call: `df.head()`, with brackets). Each loop above becomes
one line.

In [8]:
import pandas as pd

df = pd.DataFrame(orders)
df.shape

(6, 8)

In [9]:
df.head()

,order_id,customer_state,price,freight,purchased,delivered,estimated,review_score
0,o1,SP,129.9,15.1,2,9.0,12,5.0
1,o2,RJ,49.9,12.0,3,20.0,15,NaN
2,o3,SP,89.0,8.5,5,10.0,14,NaN
3,o4,MG,199.0,22.3,6,NaN,18,NaN
4,o5,SP,35.5,7.9,8,25.0,20,NaN


In [10]:
print(df["price"].sum())
print((df["customer_state"] == "SP").sum())
print(df.loc[df["delivered"] > df["estimated"], "order_id"].tolist())
df.groupby("customer_state")["price"].sum()

578.2
3
['o2', 'o5']


customer_state
MG    199.0
RJ    124.8
SP    254.4
Name: price, dtype: float64

Lists and dictionaries keep coming back inside pandas:

- a **list** of column names selects columns, a list of values feeds `isin`;
- a **dictionary** renames columns, old name → new name.

In [11]:
df[["order_id", "price"]]

,order_id,price
0,o1,129.9
1,o2,49.9
2,o3,89.0
3,o4,199.0
4,o5,35.5
5,o6,74.9


In [12]:
df[df["customer_state"].isin(["RJ", "MG"])]

,order_id,customer_state,price,freight,purchased,delivered,estimated,review_score
1,o2,RJ,49.9,12.0,3,20.0,15,NaN
3,o4,MG,199.0,22.3,6,NaN,18,NaN
5,o6,RJ,74.9,11.2,9,16.0,19,NaN


In [13]:
df.rename(columns={"freight": "freight_value"}).columns.tolist()

['order_id',
 'customer_state',
 'price',
 'freight_value',
 'purchased',
 'delivered',
 'estimated',
 'review_score']

## 2. Meet Olist

A real Brazilian marketplace: small shops sell through Olist on the big online stores. About
99,000 orders, 2016–2018, in nine related tables. We start with one: **orders**.

In [14]:
from pathlib import Path

RAW = Path("../../data/raw")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
orders.shape

(99441, 8)

In [15]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [16]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


*Observe:* every date column is `object`, i.e. text. pandas does not guess dates for us.
`describe()` summarises the numeric columns only, and there are none.

In [17]:
orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


The nine tables, and what one row of each means:

| Table | One row is… | Joins on |
|---|---|---|
| orders | an order | `customer_id` |
| order_items | an item line in an order | `order_id`, `product_id`, `seller_id` |
| order_payments | a payment of an order | `order_id` |
| order_reviews | a review | `order_id` |
| customers | a customer *as recorded on one order* | `customer_id` |
| products | a product | `product_id` |
| sellers | a seller | `seller_id` |
| category translation | a category | `product_category_name` |
| geolocation | a zip prefix location (not used) | `zip_code_prefix` |

## 3. Types and selection

In [18]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=DATE_COLUMNS)
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Already loaded as text? `pd.to_datetime` converts one column after the fact.

In [19]:
pd.to_datetime(pd.Series(["2017-11-24 10:15:00", "2018-01-02 08:00:00"]))

0   2017-11-24 10:15:00
1   2018-01-02 08:00:00
dtype: datetime64[ns]

### One column, several columns

In [20]:
orders["order_status"].head(3)

0    delivered
1    delivered
2    delivered
Name: order_status, dtype: object

In [21]:
orders[["order_id", "order_status", "order_purchase_timestamp"]].head(3)

,order_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49


### `loc`: rows and columns by label

In [22]:
orders.loc[0:2, ["order_id", "order_status"]]

,order_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,delivered


### Boolean masks: filter rows with a condition

In [23]:
is_delivered = orders["order_status"] == "delivered"
is_delivered.head()

0    True
1    True
2    True
3    True
4    True
Name: order_status, dtype: bool

In [24]:
print(is_delivered.sum(), "delivered of", len(orders))
orders[is_delivered].shape

96478 delivered of 99441


(96478, 8)

In [25]:
orders[orders["order_status"].isin(["canceled", "unavailable"])].shape

(1234, 8)

*Your turn (2 min):* Black Friday week 2017 is 20–26 November. What goes in the `...`?
`orders["order_purchase_timestamp"].between(...)`

In [26]:
orders["order_purchase_timestamp"].between("2017-11-20", "2017-11-26").sum()

np.int64(2617)

Is that the whole week? `"2017-11-26"` means midnight, so almost all of Sunday is missing.

In [27]:
black_friday_week = orders["order_purchase_timestamp"].between("2017-11-20", "2017-11-27")
black_friday_week.sum()

np.int64(3008)

In [28]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 4. Missing values are information

In [29]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

2,965 orders have no delivery date. Are they broken rows, or is something else going on?

In [30]:
pd.crosstab(orders["order_status"], orders["order_delivered_customer_date"].isna())

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


*Observe:* almost every missing delivery date belongs to an order that was never delivered.
The `NaN` is telling us the status. Only 8 "delivered" orders lack a date.

What does a blanket `dropna()` do?

In [31]:
print(len(orders), "→", len(orders.dropna()))

99441 → 96461


2,980 rows gone, for reasons we didn't choose: a missing `order_approved_at` is enough. Say
*which* column must be present:

In [32]:
delivered = orders[orders["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"])
print(len(delivered))

96470


### Derived columns

In [33]:
delivered = delivered.assign(
    delivery_days=(delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
)
delivered["delivery_days"].describe()

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

**How do you know this number is right?** "Late" means delivered after the estimated date.
But `order_estimated_delivery_date` is a *date* (midnight) and the delivery is a *timestamp*.

In [34]:
late_by_timestamp = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
late_by_day = delivered["order_delivered_customer_date"].dt.normalize() > delivered["order_estimated_delivery_date"]
print(f"late, comparing timestamps: {late_by_timestamp.mean():.1%}")
print(f"late, comparing days:       {late_by_day.mean():.1%}")

late, comparing timestamps: 8.1%
late, comparing days:       6.8%


An order delivered at 15:00 on its estimated day: late or on time? The code ran both ways. Only
a business decision tells us which number to report. We use **days**.

In [35]:
delivered = delivered.assign(is_late=late_by_day)
delivered.sort_values("delivery_days", ascending=False)[
    ["order_id", "order_purchase_timestamp", "order_delivered_customer_date", "delivery_days", "is_late"]
].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days,is_late
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,209,True
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,208,True
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,195,True
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,194,True
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,194,True


## Next week

What is *one row* of each table? We save the typed orders to the **clean** layer and meet the
first join that silently adds 4 million BRL of revenue.